# Session 1 notebook: market efficiency and factor models, with Micron

We use one stock, Micron (MU), to see the main ideas of Session 1 in data.

| Part | Question | Lecture link |
|---|---|---|
| 1 | What are the three kinds of market efficiency? | Market efficiency |
| A | Does yesterday's Micron return predict today's? (weak form) | Exercise 1 |
| B | After earnings, does the price jump at once or keep drifting? (semi-strong form) | Exercise 1 |
| C | How much of Micron's return is the market? How stable is its beta? | CAPM |
| D | What do more factors (size, value, momentum) add? | Fama-French and Carhart |

Run the cells from top to bottom. Each part ends with **How to read the output**, then you answer the questions at the end.

Data: Yahoo Finance (Micron prices and earnings) and Ken French's data library (factors). This is for learning, not trading advice.

> **A note on how this notebook was made.** It was written by an AI (Claude, from Anthropic), working from your instructor's lecture plan. Treat it the way you would treat any code you did not write: run it, read it, and check its claims against the output you see. If something looks wrong, tell your instructor. At the end, there is a question about what this means for you.

# Part 1. Market efficiency: three kinds

Eugene Fama's taxonomy asks one question: **what information is already in prices?** Each form includes the one before it.

| Form | Prices already reflect | So you should not be able to profit from | Tested below |
|---|---|---|---|
| **Weak** | All past price information: past prices and returns | Patterns in past prices, such as yesterday's return predicting today's | Part A |
| **Semi-strong** | All public information (earnings, news, filings), absorbed essentially immediately | Trading on public news after it is released | Part B |
| **Strong** | All information, including private (insider) information | Even trading on inside information | Not tested here: it needs private information |

**A more realistic middle ground (Grossman and Stiglitz, 1980).** Information is costly, so informed traders must earn something or nobody would gather it. Markets are mostly, but not completely, efficient.

**The joint hypothesis problem.** Every test of efficiency also tests a model of expected returns. If we find a pattern, either the market is inefficient or our model of "normal" returns is wrong. Part A assumes expected returns are roughly constant. Parts C and D use richer models.

**What to look for**

- **Weak form:** an autocorrelation close to zero, a small and statistically insignificant coefficient, and an R² near zero.
- **Semi-strong form:** a jump at the announcement, in the direction of the surprise, is expected either way. The question is whether the price **keeps drifting** in that direction over the following days. That is post-earnings-announcement drift (Ball & Brown, 1968), a classic violation of semi-strong efficiency.

**Two cautions.** Finding no pattern does not prove markets are efficient. Finding one does not mean you could profit from it after costs.

# Part A. Weak form: does yesterday's Micron return predict today's?

1. Download Micron's daily prices, 2015 to 2023, and compute returns.
2. Line up each day's return with the previous day's.
3. Compute the lag-1 autocorrelation.
4. Regress today's return on yesterday's and check the p-value.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm

# 1. Download Micron (MU) stock data from Yahoo Finance
ticker = 'MU'
# Using auto_adjust=True explicitly simplifies column structure to a single index
data = yf.download(ticker, start='2015-01-01', end='2024-01-01', auto_adjust=True)

# Handle potential MultiIndex columns returned by newer versions of yfinance
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

# 2. Calculate daily returns using 'Close' (which is adjusted because auto_adjust=True)
data['Returns'] = data['Close'].pct_change()
data = data.dropna()

# 3. Create the Lag-1 returns column
data['Lag1_Returns'] = data['Returns'].shift(1)
data = data.dropna()

# 4. Calculate Lag-1 Autocorrelation
lag1_autocorr = data['Returns'].corr(data['Lag1_Returns'])
print(f"Lag-1 Autocorrelation of daily returns: {lag1_autocorr:.4f}\n")

# 5. Perform OLS regression to see if Lag-1 predicts next day's return
# Dependent variable (Y): Today's Return
# Independent variable (X): Yesterday's Return (Lag-1)
Y = data['Returns']
X = sm.add_constant(data['Lag1_Returns'])

model = sm.OLS(Y, X).fit()
print(model.summary())

**How to read the output (our run: 2,262 trading days)**

| Measure | Value | Reading |
|---|---|---|
| Lag-1 autocorrelation | -0.039 | Close to zero |
| p-value on yesterday's return | 0.067 (t = -1.83) | Not significant at 5% |
| R² | 0.001 | Yesterday explains about 0.1% of today |

The slope is small, slightly negative and not significant. This is consistent with weak-form efficiency, which it does not prove. Even a significant -0.04 would have to beat trading costs on every trade.

# Part B. Semi-strong form: after earnings, does the price jump at once or keep drifting?

For about the last 50 earnings announcements:

- **Surprise** = reported EPS minus the analyst estimate.
- **Immediate jump** = Open(day t0) / Close(day t-1) - 1.
- **Drift** = Close(t+3) / Close(t0) - 1, the next three trading days.

Then split the events into positive and negative surprises and compare the averages. Note that Part A uses 2015 to 2023, while Part B uses the most recent ~50 announcements, through 2026.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# 1. Fetch MU earnings history using yfinance ticker object
mu_ticker = yf.Ticker('MU')

# Get historical earnings dates and surprises
earnings = mu_ticker.get_earnings_dates(limit=50)
if earnings is not None and not earnings.empty:
    # Clean the index and columns
    earnings = earnings.reset_index()
    earnings['Earnings Date'] = pd.to_datetime(earnings['Earnings Date']).dt.tz_localize(None)

    # Drop rows without actual or estimated EPS
    earnings = earnings.dropna(subset=['EPS Estimate', 'Reported EPS'])

    # Calculate raw surprise amount
    earnings['Surprise'] = earnings['Reported EPS'] - earnings['EPS Estimate']

    # Get daily price data over a wide window
    price_data = yf.download('MU', start='2014-01-01', end='2026-10-01', auto_adjust=True)
    if isinstance(price_data.columns, pd.MultiIndex):
        price_data.columns = price_data.columns.get_level_values(0)

    results_list = []

    for idx, row in earnings.iterrows():
        e_date = row['Earnings Date']
        # Find closest trading day on or after the earnings date
        trading_days = price_data.index[price_data.index >= e_date]
        if len(trading_days) < 5:
            continue

        t0 = trading_days[0] # Day of/immediately after earnings announcement
        # Find the trading day before earnings
        days_before = price_data.index[price_data.index < e_date]
        if len(days_before) == 0:
            continue
        t_minus_1 = days_before[-1]

        # Calculate immediate price jump (Open of t0 vs Close of t-1)
        open_t0 = price_data.loc[t0, 'Open']
        close_tm1 = price_data.loc[t_minus_1, 'Close']
        immediate_jump = (open_t0 - close_tm1) / close_tm1

        # Calculate 3-day post-earnings holding return (Close of t+3 vs Close of t0)
        t3 = trading_days[3]
        close_t0 = price_data.loc[t0, 'Close']
        close_t3 = price_data.loc[t3, 'Close']
        post_return = (close_t3 - close_t0) / close_t0

        results_list.append({
            'Earnings_Date': e_date.strftime('%Y-%m-%d'),
            'Reported_EPS': row['Reported EPS'],
            'EPS_Estimate': row['EPS Estimate'],
            'Surprise': row['Surprise'],
            'Immediate_Jump': immediate_jump,
            'Post_3D_Return': post_return
        })

    results_df = pd.DataFrame(results_list)

    # Classify surprises
    positive_surprises = results_df[results_df['Surprise'] > 0]
    negative_surprises = results_df[results_df['Surprise'] <= 0]

    print("=== Micron (MU) Earnings Surprise Impact Analysis ===\n")
    print(f"Average immediate jump after POSITIVE surprises: {positive_surprises['Immediate_Jump'].mean():.2%}")
    print(f"Average 3-day post return after POSITIVE surprises: {positive_surprises['Post_3D_Return'].mean():.2%}\n")
    print(f"Average immediate jump after NEGATIVE surprises: {negative_surprises['Immediate_Jump'].mean():.2%}")
    print(f"Average 3-day post return after NEGATIVE surprises: {negative_surprises['Post_3D_Return'].mean():.2%}\n")

    print("--- Sample Earnings Events Details ---")
    print(results_df.head(10).to_string(index=False))
else:
    print("Earnings history data is currently unavailable via Yahoo Finance API for this ticker.")

**How to read the output (our run)**

| Surprise | Immediate jump | Next 3 trading days |
|---|---|---|
| Positive | +2.33% | +0.16% |
| Negative | -5.84% | -4.72% |

- **Reading it:** positive surprises were priced in right away. After negative surprises, the price kept falling almost as much again over three days. That looks like underreaction to bad news.
- **The 3-day window is a choice.** Academic studies of PEAD usually track drift over 60 to 90 days. A 3-day window may be too short to see an effect even if it is real, so a "no drift" result here would be weaker evidence than it looks.
- **Before you trade it:** only a few dozen events, no significance test, a single stock and no trading costs. Fama (1998) would ask whether the result survives a different method. Your numbers may differ slightly from these (an earlier run gave +2.30% and +0.09% for positive surprises) because Yahoo's data is revised.

# Part C. The CAPM: how much of Micron is the market, and is its beta stable?

The CAPM says expected excess return is proportional to market beta: **E[R] = beta x E[RMRF]**. We estimate beta with the Jensen regression on monthly excess returns:

**r = alpha + beta x RMRF + e**

The same regression splits risk into two parts: **systematic** (beta² x Var[RMRF], which cannot be diversified away) and **idiosyncratic** (Var[e]). Expect noisy betas: one stock's monthly returns are very volatile.

First we load Micron's monthly returns and the Ken French factors. We reuse them in Part D.

In [ ]:
import pandas as pd
import yfinance as yf
import statsmodels.api as sm
import pandas_datareader.data as web
import matplotlib.pyplot as plt

# Same period as Part A: 2015 to 2023
# 1. Micron monthly returns from daily prices
daily = yf.download('MU', start='2014-12-01', end='2024-01-01', auto_adjust=True, progress=False)
if isinstance(daily.columns, pd.MultiIndex):
    daily.columns = daily.columns.get_level_values(0)
month_end = daily['Close'].groupby(daily.index.to_period('M')).last()
mu_monthly = month_end.pct_change().dropna().rename('MU')

# 2. Ken French monthly factors: market, size, value, and momentum (values are in percent, so divide by 100)
ff = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='2015-01-01', end='2023-12-31')[0] / 100
mom = web.DataReader('F-F_Momentum_Factor', 'famafrench', start='2015-01-01', end='2023-12-31')[0] / 100
mom.columns = [c.strip() for c in mom.columns]

# 3. Merge and compute Micron's excess return (return minus the risk-free rate)
monthly = pd.concat([mu_monthly, ff, mom], axis=1, join='inner').dropna()
monthly['Excess'] = monthly['MU'] - monthly['RF']
print(len(monthly), "months:", monthly.index[0], "to", monthly.index[-1])
monthly.head()

In [ ]:
# Jensen regression: Micron's excess return on the market's excess return
def jensen(df):
    return sm.OLS(df['Excess'], sm.add_constant(df[['Mkt-RF']])).fit()

full = jensen(monthly)
print(f"Full sample: alpha = {full.params['const']:.4f} (t = {full.tvalues['const']:.2f}), "
      f"beta = {full.params['Mkt-RF']:.2f} (t = {full.tvalues['Mkt-RF']:.2f}), R-squared = {full.rsquared:.2f}")

# Risk split: systematic share = beta^2 * Var(market) / Var(Micron excess return)
syst = full.params['Mkt-RF'] ** 2 * monthly['Mkt-RF'].var() / monthly['Excess'].var()
print(f"Systematic share of variance: {syst:.0%}   Idiosyncratic share: {1 - syst:.0%}\n")

# Is beta stable? Re-estimate on three non-overlapping 3-year windows
rows = []
for a, b in [('2015', '2017'), ('2018', '2020'), ('2021', '2023')]:
    r = jensen(monthly.loc[a:b])
    rows.append({'window': f'{a}-{b}', 'months': len(monthly.loc[a:b]),
                 'beta': r.params['Mkt-RF'], 'beta t-stat': r.tvalues['Mkt-RF'],
                 'beta std err': r.bse['Mkt-RF'], 'R-squared': r.rsquared})
pd.DataFrame(rows).set_index('window').round(2)

**How to read the output (our run)**

- Micron's full-sample beta is about 1.4, and the market explains roughly a third of its variance (R² about 0.32). The rest is idiosyncratic risk, which a diversified portfolio would largely remove.
- The beta in the three windows moves from about 2.0 to 1.4 to 1.2. Each window has only 36 monthly observations, and the standard errors are wide, so part of that drift may be estimation error. Compare the gap between windows with the standard errors before concluding that Micron's risk really changed.
- The lecture's rule: estimate betas on portfolios when you can.

# Part D. More factors: Fama-French and Carhart

Add the size, value and momentum factors to the single-factor regression:

**r = alpha + beta x RMRF + s x SMB + h x HML + u x UMD + e**

SMB is small minus big, HML is high minus low book-to-market, and UMD (called `Mom` in the data) is up minus down momentum. Each is a zero-investment long-short portfolio. This is the Carhart four-factor model, the standard for equity funds.

In [ ]:
X = sm.add_constant(monthly[['Mkt-RF', 'SMB', 'HML', 'Mom']])
carhart = sm.OLS(monthly['Excess'], X).fit()
print(carhart.summary().tables[1])
print(f"\nalpha = {100*carhart.params['const']:.2f}% per month (t = {carhart.tvalues['const']:.2f}),  R-squared = {carhart.rsquared:.2f}")

**How to read the output (our run)**

- **Alpha:** the intercept is whatever the four factors do not explain. Check its t-statistic before calling it skill.
- **Loadings:** a loading near 1 or above means a large exposure. Even 0.2 or 0.3 is a substantial exposure. Which loadings are statistically significant? What do the signs of SMB, HML and Mom say about the kind of stock Micron looked like over 2015 to 2023?
- **R² as a data check:** one stock on four factors usually has a low R². A diversified long-only fund usually has an R² above 90%, and a low R² for a fund often means a data error.

In [ ]:
# The factors themselves: growth of $1 in each long-short portfolio, 2015 to 2023
(1 + monthly[['Mkt-RF', 'SMB', 'HML', 'Mom']]).cumprod().plot(figsize=(8, 3.6),
    title="Cumulative return to the Fama-French and momentum factors, 2015 to 2023")
plt.ylabel("growth of $1"); plt.tight_layout(); plt.show()

# Questions for the room

1. If the weak-form test found a small but significant autocorrelation, could you trade on it profitably? What else would you need to know?
2. Why is the 3-day post-earnings window a real methodological choice, not an arbitrary number?
3. The jump is Open(t0) vs. Close(t-1). What does that assume about when Micron reports, and how would a wrong assumption bias it?
4. With about 50 events, how would you test whether drift differs between positive and negative surprises?
5. Which form of efficiency does each half of Exercise 1 test? Does "no weak-form predictability" say anything about semi-strong?
6. Micron's beta looks different in each 3-year window. What would you check before saying its risk really changed?
7. In Part D, would a statistically zero alpha mean Micron "failed to beat the market"? What does it mean?
8. An AI wrote this notebook in about 7 minutes, and anyone can run these tests in minutes. If analysis is cheap for everyone, what would an informational advantage look like when you start work? What would you do that this notebook cannot?